In [ ]:
# ============================================================================
# COMPLETE UEFA PREDICTOR SYSTEM
# Using: "UEFA Champions League Historical Dataset 1955-2023" from Kaggle
# Dataset link: https://www.kaggle.com/datasets/fardifaalam170041060/uefa-champions-league-historical-dataset-1955-2023
# ============================================================================

# ============================================================================
# CELL 1: INSTALL & IMPORT
# ============================================================================

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import poisson
from scipy.optimize import minimize
from scipy.special import softmax

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import log_loss
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

print("✅ UEFA Prediction System - Initialized")

# ============================================================================
# CELL 2: DOWNLOAD INSTRUCTIONS
# ============================================================================

print("""
📥 DOWNLOAD THE DATASET:

1. Go to: https://www.kaggle.com/datasets/fardifaalam170041060/uefa-champions-league-historical-dataset-1955-2023

2. Click "Download" (21.7 KB)

3. Extract the ZIP file - you'll get 2 CSV files:
   - UCL_Finals_1955-2023.csv
   - UCL_AllTime_Performance_Table.csv

4. Place BOTH files in the same folder as this notebook

5. Update the paths in the next cell if needed

NOTE: This dataset has AGGREGATE team statistics (total matches, wins, goals)
We'll use this to build ELO ratings and team strength features!
""")

# ============================================================================
# CELL 3: LOAD UEFA DATA
# ============================================================================

def load_uefa_aggregate_data():
    """
    Load UEFA Champions League aggregate team performance data.
    
    This Kaggle dataset has team statistics (matches, wins, draws, losses, goals)
    We'll convert this to usable features for our model.
    """
    
    print("📊 Loading UEFA Champions League data...")
    
    try:
        # Load team performance table
        team_perf = pd.read_csv('UCL_AllTime_Performance_Table.csv')
        
        print(f"✅ Loaded team performance data")
        print(f"📋 Columns: {list(team_perf.columns)}")
        print(f"🏟️  Teams: {len(team_perf)}")
        
        # Clean column names (handle any spacing issues)
        team_perf.columns = team_perf.columns.str.strip()
        
        # The dataset has: Team, M (Matches), W (Wins), D (Draws), L (Losses), Goals, Dif, Pt
        
        # Parse Goals column (format: "1076:55:00" means 1076 scored : 555 conceded)
        if 'Goals' in team_perf.columns:
            goals_split = team_perf['Goals'].str.split(':', expand=True)
            team_perf['Goals_For'] = pd.to_numeric(goals_split[0], errors='coerce').fillna(0)
            team_perf['Goals_Against'] = pd.to_numeric(goals_split[1], errors='coerce').fillna(0)
        
        # Calculate derived metrics
        team_perf['Win_Rate'] = team_perf['W'] / team_perf['M']
        team_perf['Draw_Rate'] = team_perf['D'] / team_perf['M']
        team_perf['Loss_Rate'] = team_perf['L'] / team_perf['M']
        team_perf['Goals_Per_Game'] = team_perf['Goals_For'] / team_perf['M']
        team_perf['Conceded_Per_Game'] = team_perf['Goals_Against'] / team_perf['M']
        
        # Calculate UEFA Strength Rating (0-100 scale)
        # Based on: Win rate, goal difference, total points
        team_perf['UEFA_Rating'] = (
            (team_perf['Win_Rate'] * 40) +  # Max 40 points for 100% wins
            (team_perf['Dif'] / team_perf['Dif'].max() * 30) +  # Max 30 for best goal diff
            (team_perf['Pt'] / team_perf['Pt'].max() * 30)  # Max 30 for most points
        )
        
        # Normalize to 1000-2500 scale (like ELO)
        min_rating = team_perf['UEFA_Rating'].min()
        max_rating = team_perf['UEFA_Rating'].max()
        team_perf['ELO_Rating'] = 1000 + (team_perf['UEFA_Rating'] - min_rating) / (max_rating - min_rating) * 1500
        
        print(f"\n✅ Team statistics computed")
        print(f"\nTop 10 teams by UEFA Rating:")
        print(team_perf[['Team', 'M', 'W', 'D', 'L', 'Goals_For', 'Goals_Against', 'ELO_Rating']]
              .sort_values('ELO_Rating', ascending=False).head(10).to_string(index=False))
        
        return team_perf
        
    except FileNotFoundError:
        print("❌ File not found!")
        print("\n💡 Make sure you:")
        print("   1. Downloaded the dataset from Kaggle")
        print("   2. Extracted the ZIP file")
        print("   3. Placed 'UCL_AllTime_Performance_Table.csv' in this folder")
        return None
    
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        import traceback
        traceback.print_exc()
        return None

# Load the data
team_stats = load_uefa_aggregate_data()

# ============================================================================
# CELL 4: CREATE TEAM LOOKUP DICTIONARY
# ============================================================================

if team_stats is not None:
    
    # Create team statistics dictionary for quick lookup
    team_lookup = {}
    
    for idx, row in team_stats.iterrows():
        team_name = row['Team']
        team_lookup[team_name] = {
            'elo': row['ELO_Rating'],
            'goals_for': row['Goals_For'],
            'goals_against': row['Goals_Against'],
            'matches': row['M'],
            'win_rate': row['Win_Rate'],
            'goals_per_game': row['Goals_Per_Game'],
            'conceded_per_game': row['Conceded_Per_Game'],
            'attack_strength': row['Goals_Per_Game'],
            'defense_strength': row['Conceded_Per_Game']
        }
    
    print(f"✅ Team lookup created for {len(team_lookup)} teams")
    
    # Also create alternate names mapping (for common variations)
    alternate_names = {
        'Man United': 'Manchester United',
        'Man City': 'Manchester City',
        'Bayern': 'Bayern Munich',
        'Barca': 'FC Barcelona',
        'Barcelona': 'FC Barcelona',
        'PSG': 'Paris Saint-Germain',
        'Inter': 'Inter Milan',
        'Benfica': 'SL Benfica',
        'Porto': 'FC Porto',
        'Ajax': 'AFC Ajax',
        'Dortmund': 'Borussia Dortmund',
        'Lyon': 'Olympique Lyon',
        'Marseille': 'Olympique Marseille',
        'Atletico': 'Atlético Madrid',
        'Atletico Madrid': 'Atlético Madrid',
    }
    
    print("✅ Alternate team names mapped")

# ============================================================================
# CELL 5: PREDICTION MODEL (SIMPLE POISSON BASED ON TEAM STATS)
# ============================================================================

def predict_uefa_match(home_team, away_team, competition='Champions League'):
    """
    Predict UEFA match using team statistics.
    
    Uses Poisson distribution based on historical attack/defense strength.
    """
    
    # Normalize team names
    home = alternate_names.get(home_team, home_team)
    away = alternate_names.get(away_team, away_team)
    
    # Get team stats
    home_stats = team_lookup.get(home)
    away_stats = team_lookup.get(away)
    
    if not home_stats:
        print(f"⚠️  Team not found: {home_team}")
        print(f"Available teams: {', '.join(sorted(list(team_lookup.keys())[:10]))}...")
        return None
    
    if not away_stats:
        print(f"⚠️  Team not found: {away_team}")
        return None
    
    # Calculate expected goals using team strengths
    # Home advantage factor (0.15 for UEFA)
    home_advantage = 0.15
    
    # Expected goals for home team
    lambda_home = (home_stats['attack_strength'] * away_stats['defense_strength'] * 
                   (1 + home_advantage))
    
    # Expected goals for away team
    lambda_away = (away_stats['attack_strength'] * home_stats['defense_strength'])
    
    # Adjust based on ELO difference
    elo_diff = home_stats['elo'] - away_stats['elo']
    elo_factor = 1 + (elo_diff / 1000) * 0.2  # Max 20% adjustment
    
    lambda_home *= elo_factor
    lambda_away /= elo_factor
    
    # Calculate probabilities using Poisson distribution
    max_goals = 10
    
    home_goal_probs = [poisson.pmf(i, lambda_home) for i in range(max_goals)]
    away_goal_probs = [poisson.pmf(i, lambda_away) for i in range(max_goals)]
    
    # Calculate match outcome probabilities
    prob_home = 0
    prob_draw = 0
    prob_away = 0
    
    for h in range(max_goals):
        for a in range(max_goals):
            prob = home_goal_probs[h] * away_goal_probs[a]
            if h > a:
                prob_home += prob
            elif h == a:
                prob_draw += prob
            else:
                prob_away += prob
    
    # Over/Under 2.5
    prob_over_25 = 0
    for h in range(max_goals):
        for a in range(max_goals):
            if h + a > 2.5:
                prob_over_25 += home_goal_probs[h] * away_goal_probs[a]
    
    return {
        'prob_home': prob_home,
        'prob_draw': prob_draw,
        'prob_away': prob_away,
        'prob_over_25': prob_over_25,
        'prob_under_25': 1 - prob_over_25,
        'lambda_home': lambda_home,
        'lambda_away': lambda_away,
        'exp_goals': lambda_home + lambda_away,
        'home_elo': home_stats['elo'],
        'away_elo': away_stats['elo'],
        'elo_diff': elo_diff
    }

# Test prediction
if team_stats is not None:
    print("\n🧪 Test prediction:")
    test_pred = predict_uefa_match('Real Madrid', 'Bayern Munich')
    if test_pred:
        print(f"Real Madrid vs Bayern Munich")
        print(f"Probabilities: H:{test_pred['prob_home']:.1%} D:{test_pred['prob_draw']:.1%} A:{test_pred['prob_away']:.1%}")
        print(f"Expected Goals: {test_pred['exp_goals']:.2f}")

# ============================================================================
# CELL 6: PROFESSIONAL UEFA PREDICTION GUI
# ============================================================================

# Storage
manual_matches = []
predictions_results = []

# GUI Components
paste_area = widgets.Textarea(
    value='',
    placeholder='''Paste matches here (one per line):
Champions League, Real Madrid, Bayern Munich, 2.30, 3.40, 3.10
Champions League, Manchester City, Inter Milan, 1.95, 3.60, 3.80
Europa League, Liverpool, Atalanta, 1.80, 3.70, 4.20''',
    layout=widgets.Layout(width='100%', height='180px', font_family='monospace')
)

parse_button = widgets.Button(
    description='📋 Parse Games',
    button_style='primary',
    layout=widgets.Layout(width='150px', height='40px')
)

clear_button = widgets.Button(
    description='🗑️ Clear',
    button_style='warning',
    layout=widgets.Layout(width='100px', height='40px')
)

predict_button = widgets.Button(
    description='🔮 PREDICT ALL',
    button_style='success',
    layout=widgets.Layout(width='250px', height='50px')
)

min_prob_slider = widgets.FloatSlider(
    value=0.35,
    min=0.20,
    max=0.70,
    step=0.05,
    description='Min Prob:',
    readout_format='.0%',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

min_ev_slider = widgets.FloatSlider(
    value=0.00,
    min=0.00,
    max=0.10,
    step=0.01,
    description='Min EV:',
    readout_format='.0%',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

status_output = widgets.Output()
results_output = widgets.Output()

# Functions
def parse_games(b):
    """Parse pasted games."""
    global manual_matches
    manual_matches = []
    
    with status_output:
        clear_output()
        
        text = paste_area.value.strip()
        if not text:
            print("❌ No games pasted!")
            return
        
        lines = text.split('\n')
        parsed = 0
        
        for line in lines:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            
            try:
                parts = [p.strip() for p in line.split(',')]
                if len(parts) != 6:
                    continue
                
                manual_matches.append({
                    'competition': parts[0],
                    'home': parts[1],
                    'away': parts[2],
                    'odds_home': float(parts[3]),
                    'odds_draw': float(parts[4]),
                    'odds_away': float(parts[5])
                })
                parsed += 1
                
            except:
                continue
        
        print(f"✅ Parsed {parsed} matches")
        
        if parsed > 0:
            print(f"\n📋 Ready to predict:")
            for m in manual_matches:
                print(f"   • {m['home']} vs {m['away']} ({m['competition']})")

def clear_paste(b):
    """Clear paste area."""
    paste_area.value = ''
    global manual_matches
    manual_matches = []
    
    with status_output:
        clear_output()
        print("🗑️ Cleared")
    
    with results_output:
        clear_output()

def predict_all_matches(b):
    """Generate predictions for all matches."""
    global predictions_results
    predictions_results = []
    
    with results_output:
        clear_output()
    
    if not manual_matches:
        with status_output:
            print("❌ No matches! Parse games first.")
        return
    
    min_prob = min_prob_slider.value
    min_ev = min_ev_slider.value
    
    with results_output:
        print("=" * 80)
        print("🏆 UEFA CHAMPIONS LEAGUE PREDICTIONS")
        print("=" * 80)
        
        value_count = 0
        
        for idx, match in enumerate(manual_matches, 1):
            comp = match['competition']
            home = match['home']
            away = match['away']
            odds = np.array([match['odds_home'], match['odds_draw'], match['odds_away']])
            
            print(f"\n{'=' * 80}")
            print(f"Match {idx}: {home} vs {away}")
            print(f"{'=' * 80}")
            print(f"Competition: {comp}")
            print(f"Odds: H:{odds[0]:.2f} D:{odds[1]:.2f} A:{odds[2]:.2f}")
            
            try:
                # Get prediction
                pred = predict_uefa_match(home, away, comp)
                
                if pred is None:
                    print("❌ Prediction failed - check team names")
                    continue
                
                # Display probabilities
                print(f"\n📊 PROBABILITIES:")
                print(f"   🏠 Home Win: {pred['prob_home']:.1%}")
                print(f"   🤝 Draw:     {pred['prob_draw']:.1%}")
                print(f"   ✈️  Away Win: {pred['prob_away']:.1%}")
                
                print(f"\n⚽ EXPECTED GOALS:")
                print(f"   Total: {pred['exp_goals']:.2f}")
                print(f"   Home: {pred['lambda_home']:.2f} | Away: {pred['lambda_away']:.2f}")
                
                print(f"\n🏆 TEAM STRENGTH:")
                print(f"   {home} ELO: {pred['home_elo']:.0f}")
                print(f"   {away} ELO: {pred['away_elo']:.0f}")
                print(f"   ELO Difference: {pred['elo_diff']:+.0f}")
                
                # Calculate EV (remove bookmaker margin)
                implied_probs = 1 / odds
                overround = implied_probs.sum()
                true_market_probs = implied_probs / overround
                
                probs = np.array([pred['prob_home'], pred['prob_draw'], pred['prob_away']])
                edges = probs - true_market_probs
                evs = (probs * odds) - 1
                
                print(f"\n💰 VALUE ANALYSIS:")
                print(f"   Bookmaker Margin: {(overround - 1) * 100:.1f}%")
                
                # Find value bets
                outcomes = ['HOME', 'DRAW', 'AWAY']
                found_value = False
                
                for i in range(3):
                    if probs[i] > min_prob and evs[i] > min_ev:
                        print(f"\n✅ VALUE BET: {outcomes[i]}")
                        print(f"   Odds: {odds[i]:.2f}")
                        print(f"   Probability: {probs[i]:.1%}")
                        print(f"   Edge: {edges[i]:.2%}")
                        print(f"   EV: {evs[i]:+.1%}")
                        
                        # Kelly stake
                        kelly = (probs[i] * (odds[i] - 1) - (1 - probs[i])) / (odds[i] - 1)
                        kelly = max(0, kelly) * 0.20
                        kelly = min(kelly, 0.03)
                        print(f"   💵 Suggested Stake: {kelly*100:.1f}% of bankroll")
                        
                        found_value = True
                        value_count += 1
                
                if not found_value:
                    best_idx = np.argmax(evs)
                    print(f"\n⚠️  NO VALUE BETS")
                    print(f"   Best: {outcomes[best_idx]} (EV: {evs[best_idx]:.1%})")
                
                # Over/Under
                print(f"\n🎲 OVER/UNDER 2.5:")
                print(f"   Over 2.5: {pred['prob_over_25']:.1%}")
                print(f"   Under 2.5: {pred['prob_under_25']:.1%}")
                
            except Exception as e:
                print(f"\n❌ Error: {e}")
                import traceback
                traceback.print_exc()
        
        print(f"\n{'=' * 80}")
        print(f"📊 SESSION SUMMARY")
        print(f"{'=' * 80}")
        print(f"Total Matches: {len(manual_matches)}")
        print(f"Value Bets Found: {value_count}")
        print(f"Hit Rate: {value_count/len(manual_matches)*100:.0f}%" if manual_matches else "N/A")
        print(f"{'=' * 80}")

# Bind buttons
parse_button.on_click(parse_games)
clear_button.on_click(clear_paste)
predict_button.on_click(predict_all_matches)

# Display GUI
display(HTML('''
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
            padding: 25px; border-radius: 12px; margin-bottom: 20px;">
    <h2 style="color: white; margin: 0;">🏆 UEFA CHAMPIONS LEAGUE PREDICTOR</h2>
    <p style="color: #f0f0f0; margin: 5px 0 0 0;">Based on Historical Team Performance Data (1955-2023)</p>
</div>
'''))

display(HTML('''
<div style="background: #fff3cd; padding: 15px; border-radius: 5px; margin-bottom: 15px;">
    <strong>📋 FORMAT:</strong> Competition, Home Team, Away Team, Home Odds, Draw Odds, Away Odds<br>
    <strong>Example:</strong> <code>Champions League, Real Madrid, Bayern Munich, 2.30, 3.40, 3.10</code><br><br>
    <strong>⚠️ Team Names Must Match Dataset:</strong><br>
    Use exact names like: "Real Madrid", "Bayern Munich", "FC Barcelona", "Manchester United", "Liverpool FC"
</div>
'''))

display(HTML("<h3>📋 Paste Matches Here:</h3>"))
display(paste_area)

button_box = widgets.HBox([parse_button, clear_button])
display(button_box)
display(HTML("<br>"))
display(status_output)

display(HTML("<br><h3>⚙️ Filters:</h3>"))
display(min_prob_slider)
display(min_ev_slider)

display(HTML("<br>"))
display(predict_button)

display(HTML("<br><hr><h3>🎯 Predictions:</h3>"))
display(results_output)

print("\n✅ UEFA Prediction System Ready!")
print("📋 Paste your matches and click 'Parse Games' to start")